# SIG Spectra Visualization

This notebook loads processed SVC `.sig` files, visualizes individual spectra, and plots mean/median curves to support quick QA before deeper analysis.

In [ ]:
!pip install specdal

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]  # one level up from notebooks/
sys.path.insert(0, str(project_root))

from processor import SigSpectraAverager, GroupSpec


## Load processed spectra into a SpecDAL Collection

In [ ]:
from specdal import Collection, Spectrum, read
from pathlib import Path
processed_dir = Path("<REPO_ROOT>/pipeline_outputs/weekly_data")
collection = Collection(name=processed_dir.name or 'processed_spectra')
collection.read(directory=str(processed_dir))
print(f"Loaded {len(collection.spectra)} spectra from {processed_dir}")


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 6))
for spectrum in collection.spectra:
    measurement = spectrum.measurement
    ax.plot(measurement.index, measurement.values, alpha=0.3)

ax.set_title('Processed SVC spectra')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Reflectance')
ax.grid(alpha=0.2)
plt.show()


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
processed_spectra_dir = "<REPO_ROOT>/pipeline_outputs/sig_resampled"
PROCESSED_SPECTRA_CSV_PATHS = {
    'bottom': "YOUR PATH HERE",

}
merged_df = pd.read_csv(PROCESSED_SPECTRA_CSV_PATHS['bottom']).iloc[1:] #remove reference scan



merged_df.head()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_resampled_spectra(df, sample_col="sample_name", max_samples=None):
    """
    Plot spectra from a wide dataframe where each row is a sample and spectral
    columns are labeled with wavelengths (e.g., '350' or 'X350').

    Args:
        df (pd.DataFrame): resampled spectra table.
        sample_col (str): column holding the sample names/ids.
        max_samples (int | None): optional limit on the number of lines to plot.
    """
    if sample_col not in df.columns:
        raise KeyError(f"Column '{sample_col}' not found.")

    spectral_cols = [
        col for col in df.columns
        if str(col).lower() not in {sample_col.lower(), "sample"}
    ]

    band_pairs = []
    for col in spectral_cols:
        cleaned = str(col).lstrip("Xx")
        if cleaned.isdigit():
            band_pairs.append((int(cleaned), col))

    if not band_pairs:
        raise ValueError("No spectral columns detected (expected numeric or X-prefixed names).")

    band_pairs.sort(key=lambda pair: pair[0])
    wavelengths = [w for w, _ in band_pairs]
    ordered_cols = [col for _, col in band_pairs]

    fig, ax = plt.subplots(figsize=(12, 6))
    rows = df if max_samples is None else df.head(max_samples)

    for _, row in rows.iterrows():
        sample = row[sample_col]
        reflectance = row[ordered_cols].to_numpy(dtype=float)
        ax.plot(wavelengths, reflectance, label=sample)

    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Reflectance")
    ax.set_title("Resampled Spectra by Sample")
    if len(rows) <= 10:
        ax.legend(loc="upper right", fontsize="small")
    else:
        ax.legend(loc="upper right", fontsize="small", ncol=2)

    plt.show()


In [ ]:
plot_resampled_spectra(merged_df, sample_col="sample_name", max_samples=20)